## Example 1: itsdangerous Serializer functionality test 1

In [ ]:
from itsdangerous import Serializer, BadSignature

# Example 1: Basic serialization and deserialization with Serializer
print("Example 1: Basic cryptographic serialization")

# Create a serializer with a secret key
secret_key = 'my-secret-key-2023'
s = Serializer(secret_key)

# Data to sign and serialize
user_data = {
    'user_id': 12345,
    'username': 'alice_doe',
    'role': 'admin',
    'permissions': ['read', 'write', 'delete']
}

print("Original data:")
print(user_data)

# Serialize and sign the data
signed_token = s.dumps(user_data)
print(f"\nSigned token: {signed_token}")
print(f"Token length: {len(signed_token)} characters")

# Verify and deserialize the data
try:
    verified_data = s.loads(signed_token)
    print(f"\nVerified data: {verified_data}")
    print(f"Data integrity confirmed: {verified_data == user_data}")
except BadSignature:
    print("Invalid signature detected!")

print("Basic serialization completed successfully")

## Example 2: itsdangerous URLSafeSerializer functionality test 2

In [ ]:
from itsdangerous import URLSafeSerializer

# Example 2: URL-safe token generation with URLSafeSerializer
print("Example 2: URL-safe token generation")

# Create URL-safe serializer
secret_key = 'url-safe-secret-2023'
url_serializer = URLSafeSerializer(secret_key)

# Session data for web applications
session_data = {
    'session_id': 'sess_98765',
    'user_id': 456,
    'csrf_token': 'csrf_abc123xyz',
    'expires_at': '2024-01-01T00:00:00Z'
}

print("Session data:")
print(session_data)

# Generate URL-safe token
url_token = url_serializer.dumps(session_data)
print(f"\nURL-safe token: {url_token}")
print(f"Token is URL-safe: {url_token.replace('-', '').replace('_', '').isalnum()}")

# Verify token integrity
verified_session = url_serializer.loads(url_token)
print(f"\nVerified session data: {verified_session}")

# Test URL encoding compatibility
import urllib.parse
encoded_token = urllib.parse.quote(url_token)
print(f"URL encoded token: {encoded_token}")
print(f"Tokens match: {url_token == encoded_token}")

print("URL-safe token generation completed successfully")

## Example 3: itsdangerous TimedSerializer functionality test 3

In [ ]:
from itsdangerous import TimedSerializer, SignatureExpired
import time

# Example 3: Time-based token expiration with TimedSerializer
print("Example 3: Time-based token expiration")

# Create timed serializer
secret_key = 'timed-secret-key'
timed_serializer = TimedSerializer(secret_key)

# Password reset token data
reset_data = {
    'email': 'user@example.com',
    'reset_type': 'password_recovery',
    'ip_address': '192.168.1.100'
}

print("Password reset data:")
print(reset_data)

# Create timed token
timed_token = timed_serializer.dumps(reset_data)
print(f"\nTimed token: {timed_token}")

# Verify token immediately (should work)
try:
    verified_data = timed_serializer.loads(timed_token, max_age=3600)  # 1 hour
    print(f"Token verified: {verified_data}")
except SignatureExpired:
    print("Token has expired!")

# Test with very short expiration
print(f"\nTesting with 2-second expiration...")
time.sleep(1)  # Wait 1 second

try:
    # This should still work
    verified_data = timed_serializer.loads(timed_token, max_age=5)
    print("Token still valid")
except SignatureExpired:
    print("Token expired within 5 seconds")

print("Time-based token expiration completed successfully")

## Example 4: itsdangerous Signer functionality test 4

In [ ]:
from itsdangerous import Signer, BadSignature

# Example 4: Simple string signing with Signer
print("Example 4: Simple string signing")

# Create a signer for simple string signing
secret_key = 'string-signing-key'
signer = Signer(secret_key)

# Test data - simple strings
test_strings = [
    'user123',
    'admin@example.com',
    'session_token_abc',
    'api_key_xyz789'
]

print("Testing string signing:")
for test_string in test_strings:
    # Sign the string
    signed_string = signer.sign(test_string)
    print(f"Original: '{test_string}'")
    print(f"Signed: '{signed_string}'")
    
    # Verify the signature (this signer just signed it: a BadSignature here fails the cell)
    unsigned_string = signer.unsign(signed_string)
    # measured (itsdangerous 2.2.0): unsign returns BYTES, so the round trip compares against the encoded input
    assert unsigned_string == test_string.encode(), (test_string, unsigned_string)
    print(f"Verified: '{unsigned_string}' ✓")
    print(f"Match: {test_string.encode() == unsigned_string}")
    
    print("-" * 40)

print("Simple string signing completed successfully")

## Example 5: itsdangerous signature verification functionality test 5

In [ ]:
from itsdangerous import URLSafeSerializer, BadSignature, BadPayload

# Example 5: Signature verification and error handling
print("Example 5: Signature verification and tamper detection")

# Create serializer for testing
secret_key = 'verification-test-key'
serializer = URLSafeSerializer(secret_key)

# Original valid data
original_data = {'message': 'Hello, secure world!', 'id': 42}
valid_token = serializer.dumps(original_data)

print(f"Valid token: {valid_token}")

# Test cases for verification
test_cases = [
    ("Valid token", valid_token),
    ("Tampered token", valid_token[:-5] + "XXXXX"),
    ("Wrong key token", URLSafeSerializer('wrong-key').dumps(original_data)),
    ("Malformed token", "invalid.token.format")
]

print("\nTesting signature verification:")
for description, token in test_cases:
    try:
        result = serializer.loads(token)
        print(f"✓ {description}: {result}")
    except BadSignature as e:
        print(f"✗ {description}: Bad signature - {type(e).__name__}")
    except BadPayload as e:
        print(f"✗ {description}: Bad payload - {type(e).__name__}")
    except Exception as e:
        print(f"✗ {description}: {type(e).__name__}")
        raise

print("Signature verification completed successfully")